In [5]:
# Reading
# Use latest mention of review
# Negative Dates

In [4]:
import pandas as pd
from pymongo import MongoClient
from sklearn.model_selection import train_test_split
import json

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

In [5]:
def read_collection() -> pd.DataFrame:
    """
    Reads all records from the configured MongoDB collection and returns them as a pandas DataFrame.
    The MongoDB connection details: database name and collection name are read from the config.json file.

    @returns:
        pd.DataFrame: Pandas DataFrame containing all records retrieved from the MongoDB collection.
    """
    # Reading configurations file to extract database name and collection names
    with open("config.json", "r") as file:
        config_data = json.load(file)

    # Establishing connection with MongoDB & reading customer reviews
    client = MongoClient(config_data['mongo_url'])
    db = client[config_data['database_name']]
    collection = db[config_data['collection_name']]
    records = list(collection.find({}))

    return pd.DataFrame(records)


# Importing dataset
raw_df = read_collection()
raw_df.head()

,_id,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,6a8093e50150837abb470a75,8fe7e2d3-e2fa-4eed-9cb8-d1a22378c7d7,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,one of the worst payment and refund system the...,1,0,None,2026-08-14 21:57:33,"Hi there, extremely sorry for the kind of expe...",2026-08-14 22:09:36,None
1,6a8093e50150837abb470a76,754a62b3-ccc3-4648-8dc6-f463f6a2f025,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,very good sir,5,0,18.19.0,2026-08-14 21:57:16,"Hi there, thank you for the brightening 5 star...",2026-08-14 22:03:41,18.19.0
2,6a8093e50150837abb470a77,ff198273-30a2-4e46-b92f-0be0f4af1a2e,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"Very nice app, am amazing, really delive.The i...",5,0,None,2026-08-14 21:56:33,"Hi there, thank you for the brightening 5 star...",2026-08-14 22:03:39,None
3,6a8093e50150837abb470a78,11d2502d-8846-4c6f-b869-e15245c9b755,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,very useful,5,0,18.15.0,2026-08-14 21:56:08,"Hi there, thank you for the brightening 5 star...",2026-08-14 22:03:38,18.15.0
4,6a8093e50150837abb470a79,37f7b890-601c-481f-a329-191e5227faf4,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,poor service,2,0,18.15.0,2026-08-14 21:55:21,"Hi there, extremely sorry for the kind of expe...",2026-08-14 22:06:33,18.15.0


# Data Preprocessing

In [6]:
# 1. Converting "at" and "repliedAt" to datetime
raw_df['at_converted'] = pd.to_datetime(pd.to_datetime(raw_df["at"]).dt.date)
raw_df["repliedAt_converted"] = pd.to_datetime(pd.to_datetime(raw_df["repliedAt"]).dt.date)

In [7]:
# 2. Calculate Response Time
raw_df['response_time'] = (pd.to_datetime(raw_df['repliedAt_converted']) - pd.to_datetime(raw_df['at_converted'])).dt.days
mask = (raw_df["repliedAt"] - raw_df["at"]).dt.total_seconds() < 0    # Identify rows with negative response time
raw_df.loc[mask, ["at", "repliedAt"]] = (raw_df.loc[mask, ["repliedAt", "at"]].to_numpy())  # Swap 'at' and 'repliedAt' for those rows
raw_df["response_time"] = ((raw_df["repliedAt"] - raw_df["at"]).dt.days)

# Annotation Sampling



<b><u>ANNOTATION STRATEGY</u></b>

Dataset: 20,000 reviews (20% of 100,000 total records) selected for annotation

Use Cases:
1. Sentiment Polarity Classification
    - Classify reviews into sentiment categories (positive, negative, neutral)
    - Leverages existing 'score' field (1-5 rating) as initial guidance

2. Complaint Extraction
    - Identify and extract complaint statements from review content
    - Flag problematic areas (delivery, pricing, quality, refunds, etc.)
    - Useful for issue tracking and customer service improvements

3. Aspect Triplet Extraction
    - Extract (aspect, opinion, sentiment) tuples from reviews
    - Example: (delivery_time, slow, negative) from "delivery is slow"
    - Enables fine-grained analysis of specific product/service attributes

Annotation Approach: LLM-Assisted with Human-in-the-Loop (HITL)
- LLM generates initial annotations using zero/few-shot prompts
- Human annotators review, correct, and validate LLM outputs
- Iterative refinement improves annotation quality and consistency
- Reduces manual annotation burden while maintaining accuracy

Workflow:
1. Annotation Phase (Current) → 20,000 reviews annotated
2. Train/Test Split → After annotation completion
3. Modeling Phase → Train models on annotated data
4. Evaluation → Test on held-out test set

Benefits:
- Balanced dataset (20% annotation rate is standard for ML projects)
- Quality labels through human validation
- Scalable approach for future data annotation

In [18]:
# Using a 20% for annotation
annotation_df , remaining_df = train_test_split(raw_df ,test_size = 0.80, stratify = raw_df['score'], random_state = 42)
annotation_df['score'].value_counts(normalize=True).sort_index() * 100

score
1    15.320
2     2.365
3     4.290
4    10.945
5    67.080
Name: proportion, dtype: float64

# Storing in Collections

In [19]:
# Establish Connection with MongoDB
client = MongoClient("mongodb://localhost:27017/")

# Select the database and collection
db = client["DAP_Blinkit"]
collection = db["AnnotationData"]

# Convert DataFrame rows into a list of dictionaries
records = annotation_df.astype(object).where(pd.notna(annotation_df), None).to_dict(orient="records")
collection.insert_many(records)

# Close the MongoDB connection
client.close()

In [20]:
# Establish Connection with MongoDB
client = MongoClient("mongodb://localhost:27017/")

# Select the database and collection
db = client["DAP_Blinkit"]
collection = db["InferenceData"]

# Convert DataFrame rows into a list of dictionaries
records = remaining_df.astype(object).where(pd.notna(remaining_df), None).to_dict(orient="records")
collection.insert_many(records)

# Close the MongoDB connection
client.close()